In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# Generate reproducible synthetic data
X, y = make_classification(
    n_samples=1200,
    n_features=4,
    n_informative=4,
    n_redundant=0,
    class_sep=1.2,
    random_state=42
)

# Create a DataFrame
data = pd.DataFrame(
    X,
    columns=[
        "usage_score",
        "tenure_score",
        "support_score",
        "engagement_score"
    ]
)

# Create a synthetic group for later fairness analysis
data["customer_group"] = np.where(
    data["tenure_score"] >= data["tenure_score"].median(),
    "Group A",
    "Group B"
)

data["target"] = y

features = [
    "usage_score",
    "tenure_score",
    "support_score",
    "engagement_score"
]

# Create reproducible training and test sets
train_data, test_data = train_test_split(
    data,
    test_size=0.25,
    random_state=42,
    stratify=data["target"]
)

# Build the preprocessing and training pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

# Train the pipeline
pipeline.fit(
    train_data[features],
    train_data["target"]
)

# Evaluate the model
predictions = pipeline.predict(
    test_data[features]
)

probabilities = pipeline.predict_proba(
    test_data[features]
)[:, 1]

accuracy = accuracy_score(
    test_data["target"],
    predictions
)

roc_auc = roc_auc_score(
    test_data["target"],
    probabilities
)

print("Training observations:", len(train_data))
print("Test observations:", len(test_data))
print(f"Accuracy: {accuracy:.3f}")
print(f"ROC-AUC: {roc_auc:.3f}")

Training observations: 900
Test observations: 300
Accuracy: 0.773
ROC-AUC: 0.802


In [3]:
import mlflow
import mlflow.sklearn

# Configure the local MLflow tracking database.
mlflow.set_tracking_uri(
    "sqlite:///mlflow.db"
)

# Create or select the experiment.
mlflow.set_experiment(
    "AISP_SG10_MLOps"
)

# Start an MLflow experiment run.
with mlflow.start_run(
    run_name="baseline_logistic_pipeline"
) as active_run:

    # Record model parameters.
    mlflow.log_params({
        "model_type": "LogisticRegression",
        "random_state": 42
    })

    # Record evaluation metrics.
    mlflow.log_metrics({
        "accuracy": accuracy,
        "roc_auc": roc_auc
    })

    # Store and register the trained pipeline.
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="classification_pipeline",
        registered_model_name=(
            "AISP_SG10_Classifier"
        )
    )

    print(
        "Experiment tracking completed."
    )

    print(
        "Run ID:",
        active_run.info.run_id
    )

    print(
        "Registered model:",
        "AISP_SG10_Classifier"
    )

    print(
        "Model URI:",
        model_info.model_uri
    )

Experiment tracking completed.
Run ID: da9d9561c1ef4cbca8134cb9fec205e2
Registered model: AISP_SG10_Classifier
Model URI: models:/m-7f02dc292df44c4ab35ca8fb79a97dc8


Successfully registered model 'AISP_SG10_Classifier'.
Created version '1' of model 'AISP_SG10_Classifier'.


In [ ]:
import sys
from importlib.metadata import version

print("Python:", sys.version.split()[0])
print("pandas:", version("pandas"))
print("scikit-learn:", version("scikit-learn"))
print("MLflow:", version("mlflow"))
print("Jupyter Notebook:", version("notebook"))